<a href="https://colab.research.google.com/github/Godstouch/GNN-Student-Risk-Prediction-/blob/main/Graph_experiment_3(relabelled).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

data = torch.load('content/real_graph_corrected_v2.pt', weights_only=False)
X = data.x  # [1000, 126], already cleanly encoded (one-hot as 0/1, ordinals standardized)
N, D = X.shape


driver_cols = np.random.choice(D, size=15, replace=False)
weights = np.random.uniform(-1, 1, size=15)

print("Driver columns (indices into x):", sorted(driver_cols.tolist()))
print("Weights:", dict(zip(driver_cols.tolist(), weights.round(3).tolist())))

signal = (X[:, driver_cols].numpy() * weights).sum(axis=1)
noise = np.random.normal(0, 0.03 * signal.std(), size=N)
risk_score = signal + noise

q1, q2 = np.quantile(risk_score, [1/3, 2/3])
y_new = np.digitize(risk_score, [q1, q2])  # 0=High-Risk (lowest score), 1=Moderate, 2=Low-Risk

import collections
print("\nNew label distribution:", collections.Counter(y_new.tolist()))

data.y = torch.tensor(y_new, dtype=torch.long)

# homophily check with the new label
ei = data.edge_index
hom = (data.y[ei[0]] == data.y[ei[1]]).float().mean().item()
counts = np.bincount(y_new)
rb = ((counts / counts.sum()) ** 2).sum()
print(f"edge homophily with new label: {hom:.4f} (random baseline: {rb:.4f})")

torch.save(data, 'content/real_graph_relabeled.pt')
print("\nSaved -> real_graph_relabeled.pt")

FileNotFoundError: [Errno 2] No such file or directory: 'content/real_graph_corrected_v2.pt'